<a href="https://colab.research.google.com/github/Ankit0974/Customer-Churn-Prediction-My-first-ANN-Project-/blob/main/Optuna_Based_tryingANN_lec10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

In [ ]:
torch.manual_seed(42)

In [2]:
pip install git

ERROR: Could not find a version that satisfies the requirement git (from versions: none)
ERROR: No matching distribution found for git


In [ ]:
df = pd.read_csv('fmnist_small.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [ ]:
df.shape

(6000, 785)

In [ ]:
X = df.drop('label', axis=1).values
y = df['label'].values

In [ ]:
X.shape

(6000, 784)

In [ ]:
y.shape

(6000,)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train = X_train/255.0
X_test = X_test/255.0

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)


    def __len__(self):
        return len(self.features)

    def __getitem__(self,idx):
      return self.features[idx], self.labels[idx]

In [ ]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [ ]:
class MyFirstNN(nn.Module):
  def __init__(self, input_dim, output_dim, num_hiddden_layers, neurons_per_layer,dropout_rate):
    super().__init__()
    layer = []

    for i in range(num_hiddden_layers):
      layer.append(nn.Linear(input_dim, neurons_per_layer))
      layer.append(nn.BatchNorm1d(neurons_per_layer))
      layer.append(nn.ReLU())
      layer.append(nn.Dropout(dropout_rate))
      input_dim = neurons_per_layer

    layer.append(nn.Linear(neurons_per_layer,output_dim))

    self.model = nn.Sequential(*layer) ##unpacking the list now its individual not in a list

  def forward(self, x):
    return self.model(x)

In [ ]:
#objective Function
def objective(trial):

  #next hyperparameter values from the  search space
  num_hidden_layers = trial.suggest_int("num_hiddden_layers", 1 ,5)
  neurons_per_layer = trial.suggest_int("neurons_per_layer",8,128, step =8)
  epochs = trial.suggest_int("epochs",10,50,step=10)
  learning_rate = trial.suggest_float("learning_rate", 1e-5,1e-1, log = True)
  dropout_rate = trial.suggest_float("dropout_rate", 0.1,0.5, step = 0.1)
  batch_size = trial.suggest_categorical("batch_size",[16,32,64,128])
  optimizer_name = trial.suggest_categorical("optimizer",['Adam','SGD','RMSprop'])
  weight_decay = trial.suggest_float("weight_decay", 1e-5,1e-3, log = True)

  #data loader

  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
  # model init
  input_dim = 784
  output_dim  = 10

  model = MyFirstNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)




  # optimizer selection
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

  if optimizer_name == 'SGD':
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  elif optimizer_name == 'Adam':
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  else:
     optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  # training loop
  for epoch in range(epochs):

    for batch_features, batch_size in train_loader:

      outputs = model(batch_features)
      loss = criterion(outputs, batch_size)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  # evaluation
  model.eval()
  #evaluation code

  total = 0
  correct = 0
  with torch.no_grad():
    for batch_features, batch_labels in train_loader:
      outputs = model(batch_features)
      _, predicted = torch.max(outputs.data, 1)
      total += batch_labels.size(0)
      correct += (predicted == batch_labels).sum().item()

  accuracy = correct/total


  return accuracy

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 22.5 MB/s eta 0:00:00


In [ ]:
import optuna

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

[I 2026-05-31 07:36:08,219] A new study created in memory with name: no-name-a7a8234f-c5c3-4e1e-8efe-294a7a109779
[I 2026-05-31 07:36:43,655] Trial 0 finished with value: 0.961875 and parameters: {'num_hiddden_layers': 2, 'neurons_per_layer': 64, 'epochs': 50, 'learning_rate': 0.001229985506222191, 'dropout_rate': 0.1, 'batch_size': 16, 'optimizer': 'Adam', 'weight_decay': 0.0005879024347879673}. Best is trial 0 with value: 0.961875.
[I 2026-05-31 07:36:49,331] Trial 1 finished with value: 0.123125 and parameters: {'num_hiddden_layers': 3, 'neurons_per_layer': 24, 'epochs': 20, 'learning_rate': 3.2370233922788004e-05, 'dropout_rate': 0.4, 'batch_size': 32, 'optimizer': 'SGD', 'weight_decay': 0.0005298319685326602}. Best is trial 0 with value: 0.961875.
[I 2026-05-31 07:37:48,737] Trial 2 finished with value: 0.8983333333333333 and parameters: {'num_hiddden_layers': 5, 'neurons_per_layer': 88, 'epochs': 50, 'learning_rate': 0.0017324007181914252, 'dropout_rate': 0.2, 'batch_size': 16, '

In [ ]:
study.best_value

0.961875

In [ ]:
study.best_params

{'num_hiddden_layers': 2,
 'neurons_per_layer': 64,
 'epochs': 50,
 'learning_rate': 0.001229985506222191,
 'dropout_rate': 0.1,
 'batch_size': 16,
 'optimizer': 'Adam',
 'weight_decay': 0.0005879024347879673}

In [ ]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [ ]:
plot_optimization_history(study).show()

In [ ]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [ ]:
# 3. Slice Plot
plot_slice(study).show()

In [ ]:
# 4. Contour Plot
plot_contour(study).show()

In [ ]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()